# Task 2 — Spam SMS Classifier
AVIP 2026 · AI/ML Engineering

**Goal:** classify an SMS as `spam` or `ham` (not spam) using TF-IDF + a classical ML model.

**Dataset:** UCI *SMS Spam Collection* (5,574 labelled messages) — free, public: https://archive.ics.uci.edu/dataset/228/sms+spam+collection  
The cell below downloads a tab-separated mirror of the same data automatically.

Run the cells **top to bottom** (Kernel → Restart & Run All).

## Cell 1 — Imports and settings

In [ ]:
import os, re, json, urllib.request
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, ConfusionMatrixDisplay)

SEED = 42   # fixed so results are reproducible
DATA_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
DATA_PATH = "../data/sms.tsv"
os.makedirs("../data", exist_ok=True); os.makedirs("../models", exist_ok=True); os.makedirs("../results", exist_ok=True)

## Cell 2 — Load the dataset
Format: one message per line → `label<TAB>message`. We drop empty rows and duplicates, then make a numeric target (ham = 0, spam = 1).

In [ ]:
if not os.path.exists(DATA_PATH):
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH, sep="\t", header=None, names=["label", "message"])
print("Rows before cleaning:", len(df))
df = df.dropna().drop_duplicates().reset_index(drop=True)
df["target"] = (df["label"] == "spam").astype(int)
print("Rows after removing duplicates:", len(df))
df["label"].value_counts()

## Cell 3 — Quick look at the data (class balance)
Spam is the minority class, so **accuracy alone is misleading** — that's why we also report precision, recall and F1.

In [ ]:
df["label"].value_counts().plot(kind="bar", color=["#3b82f6", "#ef4444"], title="Class distribution")
plt.ylabel("messages"); plt.show()
df.sample(5, random_state=SEED)

## Cell 4 — Text preprocessing
Steps: lowercase → replace URLs with `urltoken` → replace numbers with `numtoken` → remove punctuation → collapse spaces.  
(URLs and phone numbers are strong spam signals, so we keep them as tokens instead of deleting them.)

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"(https?://\S+|www\.\S+)", " urltoken ", text)
    text = re.sub(r"\d+", " numtoken ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df["clean"] = df["message"].apply(clean_text)
df[["message", "clean"]].head()

## Cell 5 — Train / test split
80% train, 20% test, **stratified** so both sets keep the same spam ratio. The test set is not touched until Cell 8.

In [ ]:
X_train, X_test, y_train, y_test, raw_train, raw_test = train_test_split(
    df["clean"], df["target"], df["message"],
    test_size=0.2, stratify=df["target"], random_state=SEED)
print("Train:", len(X_train), "| Test:", len(X_test))

## Cell 6 — Build the model pipeline and pick the best model
Pipeline = **TF-IDF** (turns text into numbers, uses 1–2 word phrases) → **classifier**.  
We compare Naive Bayes and Logistic Regression with 5-fold cross-validation on the **training data only** (no peeking at the test set).

In [ ]:
def build_pipeline(model):
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words="english")),
        ("clf", model),
    ])

candidates = {
    "Naive Bayes": MultinomialNB(alpha=0.1),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
}
cv_scores = {}
for name, m in candidates.items():
    s = cross_val_score(build_pipeline(m), X_train, y_train, cv=5, scoring="f1")
    cv_scores[name] = float(s.mean())
    print(f"{name}: cross-validated F1 = {s.mean():.4f}")

best_name = max(cv_scores, key=cv_scores.get)
print("\nBest model:", best_name)

## Cell 7 — Train the best model on the full training set

In [ ]:
model = build_pipeline(candidates[best_name]).fit(X_train, y_train)

## Cell 8 — Evaluate on the test set
Accuracy, precision, recall, F1 and the confusion matrix.

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "model": best_name,
    "cv_f1": cv_scores,
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
}
print(json.dumps(metrics, indent=2))
print(classification_report(y_test, y_pred, target_names=["ham", "spam"]))
with open("../results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["ham", "spam"]).plot(cmap="Blues", values_format="d")
plt.title(f"Confusion Matrix - {best_name} (test set)")
plt.savefig("../results/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
vals = [metrics[k] for k in ("accuracy", "precision", "recall", "f1")]
plt.figure(figsize=(5, 3.5))
plt.bar(["Accuracy", "Precision", "Recall", "F1"], vals, color="#3b82f6")
plt.ylim(0, 1.05)
for i, v in enumerate(vals): plt.text(i, v + 0.01, f"{v:.3f}", ha="center")
plt.title("Test-set metrics")
plt.savefig("../results/metrics_chart.png", dpi=150, bbox_inches="tight")
plt.show()

## Cell 9 — Example predictions with confidence (≥ 10 required)
4 real spam + 4 real ham messages from the test set, plus 4 messages I wrote myself.

In [ ]:
test_df = pd.DataFrame({"message": raw_test.values, "actual": y_test.values,
                        "predicted": y_pred, "spam_probability": y_prob})
real = pd.concat([test_df[test_df.actual == 1].head(4), test_df[test_df.actual == 0].head(4)])
real["actual"] = real["actual"].map({0: "ham", 1: "spam"})

custom_msgs = [
    "Congratulations! You won a FREE iPhone. Click http://claim-prize.com now",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: your account is suspended, call 09061234567 to verify",
    "Can you send me the notes from today's class?",
]
cprob = model.predict_proba([clean_text(m) for m in custom_msgs])[:, 1]
custom = pd.DataFrame({"message": custom_msgs, "actual": "(custom)",
                       "predicted": (cprob >= 0.5).astype(int), "spam_probability": cprob})

ex = pd.concat([real, custom], ignore_index=True)
ex["predicted_label"] = ex["predicted"].map({0: "ham", 1: "spam"})
ex["confidence"] = np.where(ex["predicted"] == 1, ex["spam_probability"], 1 - ex["spam_probability"])
ex = ex[["message", "actual", "predicted_label", "confidence", "spam_probability"]].round(4)
ex.to_csv("../results/example_predictions.csv", index=False)
ex

## Cell 10 — Save the trained model
The whole pipeline (TF-IDF + classifier) is saved in one file. Run `python src/predict.py "your text"` from the project root to use it.

In [ ]:
joblib.dump(model, "../models/spam_classifier.joblib")

# Reload and test it, exactly like a user would
loaded = joblib.load("../models/spam_classifier.joblib")
msg = "WINNER!! You have been selected for a free prize. Call now"
p = loaded.predict_proba([clean_text(msg)])[0, 1]
print(f"spam probability = {p:.3f} ->", "SPAM" if p >= 0.5 else "HAM")

## Cell 11 — Mistakes the model made
Looking at errors is how you improve a model and what to write about in the summary.

In [ ]:
errors = test_df[test_df["actual"] != test_df["predicted"]].round(3)
print("Misclassified messages:", len(errors))
errors[["message", "actual", "predicted", "spam_probability"]].head(10)